# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [1]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [2]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [3]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [4]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

75 ciudades | 47 capitales | ámbitos: 15 comunidades
Capitales de Galicia: ['A Coruña', 'Ourense', 'Lugo', 'Pontevedra']


In [5]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [6]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [7]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### Nombre del algoritmo

<div style="border-left:5px solid #4C72B0;background:#EEF2FA;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

**Funcionamiento en lenguaje llano:**
El **Algoritmo de Dantzig-Fulkerson-Johnson (DFJ)** resuelve el TSP comenzando por la versión más simple posible del problema: el **Problema de Asignación**. En esta fase inicial, solo exige que de cada ciudad se salga exactamente una vez y a cada ciudad se entre exactamente una vez, sin importar si el camino está fragmentado.

Al resolver esta simplificación, es habitual obtener soluciones "rotas" compuestas por varios subciclos cerrados aislados (llamados *subtours* o componentes conexas). Cuando esto ocurre, el algoritmo identifica uno de estos subciclos y le añade una regla matemática extra (un "corte" o restricción de conectividad) que dicta: *"OBLIGATORIAMENTE debes usar al menos una arista que conecte este subgrupo con el resto del mundo"*.

El problema de enteros se vuelve a resolver incorporando esta nueva regla acumulada. El proceso se repite iterativamente agregando restricciones a medida que la solución devuelva nuevos subtours, hasta que el resultado final sea un único ciclo continuo (camino hamiltoniano) que visite todas las ciudades.

**Pasos del algoritmo (Formulación DFJ por eliminación iterativa de subtours):**

1. **Inicialización (Modelo Base):** Formular el problema de asignación sobre un grafo de $N$ vértices:

$$\min \sum_{i=1}^{N} \sum_{j=1}^{N} c_{ij} x_{ij}$$



Sujeto a:

$$\sum_{j=1, j \neq i}^{N} x_{ij} = 1 \quad \forall i \in \{1, \dots, N\} \quad \text{(Grado de salida = 1)}$$


$$\sum_{i=1, i \neq j}^{N} x_{ij} = 1 \quad \forall j \in \{1, \dots, N\} \quad \text{(Grado de entrada = 1)}$$


$$x_{ij} \in \{0, 1\}$$


2. **Resolver PLE:** Obtener la solución óptima actual $X^*$ del modelo mediante un solver de Programación Lineal Entera.
3. **Análisis de componentes:** Identificar las componentes conexas (subtours) formadas por las aristas activas ($x_{ij} = 1$).
4. **Evaluación de parada:**
    * **Si existe una única componente conexa de tamaño $N$:** **FIN**. $X^*$ es el ciclo hamiltoniano óptimo.
    * **Si existen múltiples componentes conexas ($S_1, S_2, \dots, S_k$):** Seleccionar una de las componentes $S \subset V$ (donde $1 < \vert{}S\vert{} < N$).


5. **Generación de los cortes (Eliminación de Subtours):** Para **cada una** de las componentes conexas aisladas $S_m$ detectadas ($m = 1, \dots, k-1$), añadir al modelo su correspondiente Restricción de Eliminación de Subtour (SEC):

$$\sum_{i \in S_m} \sum_{j \notin S_m} x_{ij} \ge 1 \quad \text{(Conectividad: al menos una salida de la componente $S_m$)}$$

> *(O de forma equivalente: $\sum_{i \in S_m} \sum_{j \in S_m} x_{ij} \le \vert{}S_m\vert{} - 1$, prohibiendo formar un ciclo interno cerrado en $S_m$)*.

6. **Iteración:** Volver al **Paso 2** con el modelo enriquecido.

**Tipo:** Exacto.
  
**Complejidad en tiempo:** **Exponencial** ($\mathcal{O}(2^N)$ en el peor caso, ya que en cada iteración se resuelve un problema NP-duro y el número de subtours posibles a cortar crece exponencialmente). **en memoria:** **Exponencial** ($\mathcal{O}(2^N)$ en el peor caso teórico si se generasen todos los cortes posibles, aunque en la práctica es $\mathcal{O}(K \cdot N^2)$ en la iteración $K$).

**Propiedades:**
    * **¿Siempre encuentra el óptimo?:** Sí, garantiza la solución óptima global al ser exacto.
    * **¿Es determinista?:** Sí.
    * **¿Tiene garantía de calidad?:** Sí (garantía del 100% de optimalidad, cota de error del 0%).
    * **¿Necesita distancias simétricas?:** No; es válido tanto para TSP asimétrico (dirigido) como simétrico (no dirigido).


**¿Hasta cuántas ciudades es razonable usarlo?**
    * En su **versión pura e iterativa básica** (re-resolviendo el PLE completo desde cero en cada paso), es práctico hasta **30 – 50 ciudades**.
    * Integrado como **Lazy Constraints en Branch and Cut** con solvers modernos (como Gurobi, CPLEX o Concorde), se puede escalar hasta **decenas de miles de ciudades**.

</div>

**💻 CÓDIGO**

*[Sustituye `mi_algoritmo` por tu algoritmo, con un nombre único (por ejemplo, `fuerza_bruta`). Debe cumplir el contrato: recibe el grafo de ciudades `K` y devuelve `(ruta, coste)`. Borra este texto entre corchetes.]*

In [28]:
!python -m pip install --upgrade pulp
!python -m pip install --upgrade "pulp[cbc]"


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/135.3 MB ? eta -:--:--
    --------------------------------------- 2.9/135.3 MB 31.2 MB/s eta 0:00:05
   --- ------------------------------------ 10.2/135.3 MB 31.3 MB/s eta 0:00:04
   ----- ---------------------------------- 17.3/135.3 MB 33.1 MB/s eta 0:00:04
   ------- -------------------------------- 26.0/135.3 MB 35.0 MB/s eta 0:00:04
   ---------- ----------------------------- 35.9/135.3 MB 37.4 MB/s eta 0:00:03
   ------------- -------------------------- 44.3/135.3 MB 38.0 MB/s eta 0:00:03
   --------------- ------------------------ 52.4/135.3 MB 38.0 MB/s eta 0:00:03
   ----------------- ---------------------- 60.8/135.3 MB 38.4 MB/s eta 0:00:02
   -------------------- ------------------- 69.2/135.3 MB 38.4 MB/s eta 0:00:02
   ---------------------- ----------------- 76.5/135.3 MB 38.3 MB/s eta 0:00:02
   ------------------------- -------------- 84.7/135.3 MB 38.4 MB/s eta 0:00:02
   --------------------------- ------------ 92.8/1


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
def reconstruir_recorrido(aristas_seleccionadas, inicio):
    siguiente = {u: v for u, v in aristas_seleccionadas}

    recorrido = [inicio]
    actual = inicio

    while True:
        actual = siguiente[actual]
        recorrido.append(actual)

        if actual == inicio:
            break

    return recorrido[:-1]

In [14]:
import pulp
import networkx as nx
import matplotlib.pyplot as plt


def dfj(K,inicio):
    """
    Resuelve el TSP mediante el modelo de asignación y la
    incorporación iterativa de restricciones de conectividad.

    K: Grafo dirigido de NetworkX.
       Cada arista debe tener un atributo 'peso'.
    """

    i = 0

    # 1. Crear el modelo de minimización
    prob = pulp.LpProblem("TSP_Asignacion", pulp.LpMinimize)

    # 2. Crear las variables binarias para cada arista
    x = prob.add_variable_dicts(
        "x", list(K.edges), cat=pulp.LpBinary
    )

    # 3. Restricciones de grado: una entrada y una salida por nodo
    for u in K.nodes:

        aristas_salida = [e for e in K.edges if e[0] == u]
        prob += (
            pulp.lpSum(x[e] for e in aristas_salida) == 1,
            f"Salida_{u}"
        )

        aristas_entrada = [e for e in K.edges if e[1] == u]
        prob += (
            pulp.lpSum(x[e] for e in aristas_entrada) == 1,
            f"Entrada_{u}"
        )

    # 4. Función objetivo: minimizar el coste total
    prob += pulp.lpSum(
        K[u][v]["peso"] * x[(u, v)]
        for u, v in K.edges
    ), "Coste_Total"

    # 5. Resolver el modelo inicial
    resultado = prob.solve(pulp.COIN_CMD(msg=False))

    if not resultado.has_solution:
        print("No se ha encontrado una solución.")
        return None, []

    # 6. Obtener las aristas seleccionadas y sus componentes
    aristas_seleccionadas = [
        e for e in K.edges
        if x[e].value() is not None and x[e].value() > 0.5
    ]

    nuevoGrafo = nx.DiGraph()
    nuevoGrafo.add_nodes_from(K.nodes)
    nuevoGrafo.add_edges_from(aristas_seleccionadas)

    componentes = list(nx.weakly_connected_components(nuevoGrafo))

    # 7. Eliminar los subciclos mediante restricciones de conectividad
    while len(componentes) > 1:
        '''
        print(f"\nIteración {i}")
        print("Componentes conexos:", componentes)
        print("Coste obtenido:", resultado.objective)
        '''
        # Añadir una restricción para cada componente
        for j, S in enumerate(componentes):

            notS = set(K.nodes) - S

            # Aristas que salen de S hacia el resto del grafo
            aristas_restriccion = [
                (u, v)
                for u in S
                for v in notS
                if K.has_edge(u, v)
            ]

            prob += (
                pulp.lpSum(x[e] for e in aristas_restriccion) >= 1,
                f"Conectividad_{i}_{j}"
            )

        i += 1

        # 8. Resolver de nuevo con las restricciones añadidas
        resultado = prob.solve(pulp.COIN_CMD(msg=False))

        if not resultado.has_solution:
            print("No se ha encontrado una solución.")
            return None, []

        # 9. Actualizar las aristas seleccionadas
        aristas_seleccionadas = [
            e for e in K.edges
            if x[e].value() is not None and x[e].value() > 0.5
        ]

        # 10. Reconstruir el grafo de la solución
        nuevoGrafo = nx.DiGraph()
        nuevoGrafo.add_nodes_from(K.nodes)
        nuevoGrafo.add_edges_from(aristas_seleccionadas)

        componentes = list(
            nx.weakly_connected_components(nuevoGrafo)
        )
        # 11. Mostrar el resultado final
        coste_total = resultado.objective
    '''
    
        print("\nSolución final")
        print("Iteraciones:", i)
        print("Componentes conexos:", componentes)
        print("Estado:", resultado.status_str)
        print("Coste obtenido:", coste_total)
        print("Aristas seleccionadas:", aristas_seleccionadas)
    
        # 12. Dibujar la solución
        pos = nx.spring_layout(nuevoGrafo, seed=42)
    
        nx.draw(
            nuevoGrafo,
            pos,
            with_labels=True,
            arrows=True,
            node_size=700
        )
    
        plt.title("Solución del TSP")
        plt.show()
    '''
    print("Total de iteraciones ejecutadas:",i)
    ruta = reconstruir_recorrido(aristas_seleccionadas,inicio)
    return ruta,  coste_total

In [11]:
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
#for x,y in K.edges:
 #   print(K[x][y]["peso"])

In [12]:
dfj(K, inicio="Vigo")

Total de iteraciones ejecutadas: 1
['Vigo', 'Ourense', 'Lugo', 'A Coruña', 'Santiago de Compostela', 'Pontevedra']


(['Vigo',
  'Ourense',
  'Lugo',
  'A Coruña',
  'Santiago de Compostela',
  'Pontevedra'],
 460.438)

**Prueba rápida** con una versión pequeña (las 6 ciudades de Galicia).

*[Cambia los parámetros de `cargar_version` para probar otras versiones y borra este texto entre corchetes.]*

In [15]:
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Vigo")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(dfj, K, memoria=True, inicio="Vigo"))

Total de iteraciones ejecutadas: 1
Vigo → Ourense → Lugo → A Coruña → Santiago de Compostela → Pontevedra | coste: 460.4 km
Total de iteraciones ejecutadas: 1
Total de iteraciones ejecutadas: 1
{'algoritmo': 'dfj', 'ciudades': 6, 'coste': 460.438, 'unidad': 'km', 'segundos': 0.0763, 'memoria_mb': 0.04}


In [16]:
K = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Sevilla")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(dfj, K, memoria=True, inicio="Sevilla"))

Total de iteraciones ejecutadas: 3
Sevilla → Cádiz → Málaga → Córdoba → Jaén → Granada → Almería → Murcia → Alicante → Albacete → Cuenca → Teruel → Valencia → Castellón de la Plana → Tarragona → Barcelona → Girona → Lleida → Huesca → Zaragoza → Soria → Logroño → Pamplona → San Sebastián → Vitoria-Gasteiz → Bilbao → Santander → Burgos → Palencia → Valladolid → León → Oviedo → Lugo → A Coruña → Pontevedra → Ourense → Zamora → Salamanca → Ávila → Segovia → Guadalajara → Madrid → Toledo → Ciudad Real → Cáceres → Badajoz → Huelva | coste: 5917.6 km
Total de iteraciones ejecutadas: 3
Total de iteraciones ejecutadas: 3
{'algoritmo': 'dfj', 'ciudades': 47, 'coste': 5917.562, 'unidad': 'km', 'segundos': 0.9657, 'memoria_mb': 1.09}


In [17]:
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Vigo")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Vigo"))

Total de iteraciones ejecutadas: 3
Vigo → Ourense → Zamora → Salamanca → Ávila → Segovia → Alcobendas → Torrejón de Ardoz → Guadalajara → Alcalá de Henares → Rivas-Vaciamadrid → Madrid → Leganés → Alcorcón → Móstoles → Fuenlabrada → Getafe → Parla → Toledo → Ciudad Real → Cáceres → Badajoz → Huelva → Sevilla → Dos Hermanas → Jerez de la Frontera → Cádiz → Algeciras → Marbella → Málaga → Córdoba → Jaén → Granada → Roquetas de Mar → Almería → Cartagena → Murcia → Elche → Alicante → Albacete → Cuenca → Teruel → Valencia → Castellón de la Plana → Reus → Tarragona → L'Hospitalet de Llobregat → Barcelona → Santa Coloma de Gramenet → Badalona → Mataró → Girona → Sabadell → Terrassa → Lleida → Huesca → Zaragoza → Soria → Logroño → Pamplona → San Sebastián → Vitoria-Gasteiz → Bilbao → Barakaldo → Santander → Burgos → Palencia → Valladolid → León → Oviedo → Gijón → Lugo → A Coruña → Santiago de Compostela → Pontevedra | coste: 6146.2 km
################Rendimiento####################
Total de it

In [18]:
K = cargar_version(aristas="autovias_principales", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Vigo")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Vigo"))

Total de iteraciones ejecutadas: 5
Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Gijón → Oviedo → León → Valladolid → Palencia → Burgos → Santander → Barakaldo → Bilbao → Vitoria-Gasteiz → San Sebastián → Pamplona → Logroño → Soria → Zaragoza → Huesca → Lleida → Sabadell → Terrassa → Santa Coloma de Gramenet → Girona → Mataró → Badalona → Barcelona → L'Hospitalet de Llobregat → Tarragona → Reus → Castellón de la Plana → Valencia → Teruel → Cuenca → Albacete → Alicante → Elche → Murcia → Cartagena → Almería → Roquetas de Mar → Granada → Jaén → Córdoba → Málaga → Marbella → Algeciras → Cádiz → Jerez de la Frontera → Dos Hermanas → Sevilla → Huelva → Badajoz → Cáceres → Ciudad Real → Toledo → Parla → Getafe → Leganés → Fuenlabrada → Móstoles → Alcorcón → Madrid → Rivas-Vaciamadrid → Torrejón de Ardoz → Guadalajara → Alcalá de Henares → Alcobendas → Segovia → Ávila → Salamanca → Zamora → Ourense | coste: 6377.5 km
################Rendimiento####################
Total de it

In [19]:
K = cargar_version(aristas="solo_autovias", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Vigo")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Vigo"))

Total de iteraciones ejecutadas: 10
Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Zamora → Valladolid → Palencia → Burgos → Alcobendas → Torrejón de Ardoz → Alcalá de Henares → Soria → Guadalajara → Rivas-Vaciamadrid → Parla → Getafe → Alcorcón → Leganés → Móstoles → Madrid → Ávila → Segovia → Fuenlabrada → Toledo → Ciudad Real → Córdoba → Dos Hermanas → Huelva → Sevilla → Cáceres → Jerez de la Frontera → Cádiz → Algeciras → Marbella → Granada → Jaén → Málaga → Roquetas de Mar → Almería → Cartagena → Murcia → Elche → Alicante → Albacete → Valencia → Teruel → Castellón de la Plana → Tarragona → Reus → Barcelona → L'Hospitalet de Llobregat → Sabadell → Terrassa → Badalona → Santa Coloma de Gramenet → Mataró → Girona → Lleida → Zaragoza → Huesca → Logroño → Vitoria-Gasteiz → Pamplona → San Sebastián → Barakaldo → Bilbao → Santander → Gijón → Oviedo → León → Ourense | coste: 8509.3 km
################Rendimiento####################
Total de iteraciones ejecutadas: 10
Total

In [20]:
K = cargar_version(aristas="sin_autovias", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Vigo")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Vigo"))

Total de iteraciones ejecutadas: 5
Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Oviedo → Gijón → Santander → Burgos → Barakaldo → Bilbao → San Sebastián → Pamplona → Vitoria-Gasteiz → Logroño → Soria → Zaragoza → Huesca → Lleida → Girona → Mataró → Badalona → Santa Coloma de Gramenet → L'Hospitalet de Llobregat → Barcelona → Sabadell → Terrassa → Tarragona → Reus → Castellón de la Plana → Valencia → Teruel → Cuenca → Albacete → Elche → Alicante → Cartagena → Murcia → Jaén → Granada → Almería → Roquetas de Mar → Málaga → Marbella → Algeciras → Jerez de la Frontera → Cádiz → Córdoba → Dos Hermanas → Sevilla → Huelva → Badajoz → Cáceres → Ciudad Real → Toledo → Móstoles → Alcorcón → Leganés → Getafe → Fuenlabrada → Parla → Rivas-Vaciamadrid → Alcalá de Henares → Torrejón de Ardoz → Madrid → Alcobendas → Guadalajara → Segovia → Ávila → Salamanca → Zamora → Valladolid → Palencia → León → Ourense | coste: 8465.7 km
################Rendimiento####################
Total de it

In [22]:
K = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Sevilla")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
#print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Sevilla"))

Total de iteraciones ejecutadas: 3
################Rendimiento####################
Total de iteraciones ejecutadas: 3
Total de iteraciones ejecutadas: 3
{'algoritmo': 'dfj', 'ciudades': 47, 'coste': 5917.562, 'unidad': 'km', 'segundos': 0.9868, 'memoria_mb': 1.09}


In [23]:
K = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Madrid")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
#print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Madrid"))

Total de iteraciones ejecutadas: 3
################Rendimiento####################
Total de iteraciones ejecutadas: 3
Total de iteraciones ejecutadas: 3
{'algoritmo': 'dfj', 'ciudades': 47, 'coste': 5917.562, 'unidad': 'km', 'segundos': 0.9604, 'memoria_mb': 1.09}


In [24]:
K = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Barcelona")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
#print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Barcelona"))

Total de iteraciones ejecutadas: 3
################Rendimiento####################
Total de iteraciones ejecutadas: 3
Total de iteraciones ejecutadas: 3
{'algoritmo': 'dfj', 'ciudades': 47, 'coste': 5917.562, 'unidad': 'km', 'segundos': 0.9586, 'memoria_mb': 1.09}


In [28]:
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="andalucia", peso="km", general="grafo_dirigido")
ruta, coste = dfj(K, inicio="Sevilla")
validar_ruta(K, ruta)                            # comprueba que visita cada ciudad una vez
#print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print("################Rendimiento####################")
print(medir(dfj, K, memoria=True, inicio="Sevilla"))

Total de iteraciones ejecutadas: 2
################Rendimiento####################
Total de iteraciones ejecutadas: 2
Total de iteraciones ejecutadas: 2
{'algoritmo': 'dfj', 'ciudades': 13, 'coste': 1251.307, 'unidad': 'km', 'segundos': 0.1589, 'memoria_mb': 0.11}


<div style="border-left:5px solid #55A868;background:#EAF6ED;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

| Nº Ciudades | Iteraciones | Coste (km) | Tiempo (s) | Memoria (MB) |
| :---: | :---: | :---: | :---: | :---: |
| 6 | 1 | 460.438 | 0.0763 | 0.04 |
| 47 | 3 | 5917.562 | 0.9657 | 1.09 |
| 75 | 3 | 6146.165 | 3.3242 | 3.25 |

Se ha observado tambien que existe un ciclo halmitoniano con grafos sin autovias y con autovias principales. Pero no existe un camino halmitoniano que pase por todas las ciudades con solo autovías. (Se encontró camino, porque hay 3 ciudades muy aislada que se omitio en el cálculo). Y queda demostrado que el coste del camino no varía según la ciudad de partida
</div>

---